# FitForge evaluator prototype

SCSE4041 Milestone 2 · FIT FORGE · PE-B13-56

Run the setup, model and launch cells in order on a Colab CPU runtime. The first setup downloads about 1.1 GB for the local model. Open the app link printed by the launch cell. You can use the plan generator and ask the coach a fitness question without a FitForge account. Guest data belongs to this notebook runtime. The saved output below shows the same API case from the milestone test run.

## 1. Build the web app

In [ ]:
from pathlib import Path
import os, secrets, subprocess, time, urllib.request

base = Path("/content/fitforge-demo")
base.mkdir(exist_ok=True)
repo = base / "fit-forge"
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "milestone-2", "https://github.com/AyushKuchhal/fit-forge.git", repo], check=True)

node_url = "https://nodejs.org/dist/v24.22.3/node-v24.22.3-linux-x64.tar.xz"
node_home = base / "node-v24.22.3-linux-x64"
if not node_home.exists():
    subprocess.run(["curl", "-fLsS", "--retry", "3", node_url, "-o", base / "node.tar.xz"], check=True)
    subprocess.run(["tar", "-xJf", base / "node.tar.xz", "-C", base], check=True)

env = dict(os.environ, PATH=f"{node_home / 'bin'}:{os.environ['PATH']}")
for part in ["client", "server"]:
    subprocess.run(["npm", "ci"], cwd=repo / part, env=env, check=True)
    subprocess.run(["npm", "run", "build"], cwd=repo / part, env=env, check=True)
print("FitForge client and server built.")

## 2. Start the local coach model

In [ ]:
llama_base = base / "llama"
llama_base.mkdir(exist_ok=True)
release = "https://github.com/ggml-org/llama.cpp/releases/download/b12631/llama-b12631-bin-ubuntu-x64.tar.gz"
server_bin = next(llama_base.rglob("llama-server"), None)
if server_bin is None:
    subprocess.run(["curl", "-fLsS", "--retry", "3", release, "-o", base / "llama.tar.gz"], check=True)
    subprocess.run(["tar", "-xzf", base / "llama.tar.gz", "-C", llama_base], check=True)
    server_bin = next(llama_base.rglob("llama-server"))

model_file = base / "qwen2.5-1.5b-instruct-q4_k_m.gguf"
if not model_file.exists() or model_file.stat().st_size != 1117320736:
    temp_model = model_file.with_suffix(".part")
    subprocess.run(["curl", "-fLsS", "--retry", "3", "https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct-GGUF/resolve/main/qwen2.5-1.5b-instruct-q4_k_m.gguf", "-o", temp_model], check=True)
    assert temp_model.stat().st_size == 1117320736
    temp_model.replace(model_file)

def wait_for(url, process, logfile, seconds=120):
    for _ in range(seconds):
        if process.poll() is not None:
            break
        try:
            with urllib.request.urlopen(url, timeout=2) as result:
                if result.status == 200:
                    return
        except OSError:
            pass
        time.sleep(1)
    raise RuntimeError(Path(logfile).read_text(errors="replace")[-3000:])

previous = globals().get("fitforge_model_process")
if previous is not None and previous.poll() is None:
    previous.terminate()
    previous.wait(timeout=15)
model_log = base / "model.log"
model_env = dict(env, LD_LIBRARY_PATH=str(server_bin.parent))
with model_log.open("wb") as output:
    fitforge_model_process = subprocess.Popen(
        [server_bin, "-m", model_file, "--alias", "fitforge-coach", "-c", "2048",
         "-t", "4", "--host", "127.0.0.1", "--port", "8080"],
        cwd=server_bin.parent, env=model_env, stdout=output, stderr=subprocess.STDOUT)
wait_for("http://127.0.0.1:8080/health", fitforge_model_process, model_log)
print("Qwen coach is ready.")

## 3. Launch the browser prototype

In [ ]:
from google.colab import output

previous = globals().get("fitforge_app_process")
if previous is not None and previous.poll() is None:
    previous.terminate()
    previous.wait(timeout=15)
app_log = base / "app.log"
app_env = dict(env, PORT="4000", JWT_SECRET=secrets.token_hex(32), COACH_MODEL_URL="http://127.0.0.1:8080")
with app_log.open("wb") as log:
    fitforge_app_process = subprocess.Popen(["node", "dist/index.js"], cwd=repo / "server",
                                            env=app_env, stdout=log, stderr=subprocess.STDOUT)
wait_for("http://127.0.0.1:4000/health", fitforge_app_process, app_log)
output.serve_kernel_port_as_window(4000)

## 4. Replay the coach test

In [1]:
import json

question = "I train four days a week. How should I recover between sessions?"
payload = {"messages": [{"role": "user", "content": question}], "userContext": {"weightKg": 78}}
request = urllib.request.Request("http://127.0.0.1:4000/api/ai/chat", json.dumps(payload).encode(),
                                 {"Content-Type": "application/json"})
with urllib.request.urlopen(request, timeout=60) as reply:
    events = [item[6:] for item in reply.read().decode().splitlines() if item.startswith("data: ")]
    text = "".join(json.loads(event)["text"] for event in events if event != "[DONE]")
    print("Question:", question)
    print("Source:", reply.headers.get("X-FitForge-Coach"))
    print(text)

Question: I train four days a week. How should I recover between sessions?
Source: model
Answer: You should aim for at least 7-8 hours of sleep per night and incorporate active recovery methods such as walking or yoga to help your body recover between training sessions.
Next step: Discuss active recovery options with your coach.
